# 📚 Notebook 01: Setup e Conexão com PostgreSQL

**Projeto:** Ciência de Dados - Inhire  
**Data:** 2026-03-05  
**Objetivo:** Configurar ambiente e estabelecer conexão com banco de dados  
**Nível:** Iniciante  

---

## 🎯 O que você vai aprender neste notebook:

1. ✅ Importar bibliotecas essenciais (pandas, numpy, etc.)
2. ✅ Conectar ao PostgreSQL usando Python
3. ✅ Executar queries SQL e carregar dados em DataFrames
4. ✅ Explorar a estrutura do banco de dados Inhire
5. ✅ Criar funções helper para facilitar queries futuras

---

## 📖 Conceitos Básicos (para iniciantes)

### O que é um DataFrame?
- É uma **tabela** de dados em Python (como uma planilha Excel)
- Criado pela biblioteca **pandas**
- Tem linhas (registros) e colunas (variáveis)
- Permite fazer análises estatísticas facilmente

### O que é SQL?
- **SQL** = Structured Query Language (Linguagem de Consulta Estruturada)
- Usado para **consultar** dados de bancos de dados relacionais
- Exemplo: `SELECT * FROM vagas WHERE status = 'OPEN'`

### O que é PostgreSQL?
- É um **banco de dados relacional** (armazena dados em tabelas)
- Gratuito e open-source
- Usado para armazenar os dados do Inhire

---

## 1️⃣ Importar Bibliotecas

Vamos começar importando as bibliotecas que vamos usar:

In [ ]:
# Manipulação de dados
import pandas as pd  # Para trabalhar com DataFrames (tabelas)
import numpy as np   # Para operações numéricas

# Conexão com banco de dados
import psycopg2  # Driver para PostgreSQL
from sqlalchemy import create_engine  # ORM para facilitar conexões

# Configurações do projeto
import sys
sys.path.append('..')  # Adicionar pasta pai ao path
from config_ds import Config  # Importar configurações

# Visualização (vamos usar nos próximos notebooks)
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

# Mostrar todas as colunas do DataFrame (para não truncar)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("✓ Bibliotecas importadas com sucesso!")
print(f"✓ Pandas versão: {pd.__version__}")
print(f"✓ NumPy versão: {np.__version__}")

## 2️⃣ Carregar Configurações

Vamos usar o arquivo `config_ds.py` que criamos para centralizar configurações:

In [ ]:
# Exibir informações de configuração
Config.info()

# Verificar se conseguimos conectar ao banco
if Config.verificar_conexao_db():
    print("\n[OK] Pronto para começar!")
else:
    print("\n[ERRO] Não foi possível conectar ao banco. Verifique as credenciais no .env")

## 3️⃣ Criar Conexão com PostgreSQL

Vamos criar uma função para facilitar a execução de queries SQL:

In [ ]:
def executar_query(query: str, params: dict = None) -> pd.DataFrame:
    """
    Executa uma query SQL e retorna os resultados como DataFrame do pandas.
    
    Parâmetros:
        query (str): Query SQL a ser executada
        params (dict): Parâmetros para a query (opcional)
    
    Retorna:
        pd.DataFrame: Resultados da query
    
    Exemplo:
        df = executar_query("SELECT * FROM vagas LIMIT 10")
    """
    try:
        # Criar engine do SQLAlchemy
        engine = create_engine(Config.DATABASE_URL)
        
        # Executar query e retornar DataFrame
        df = pd.read_sql_query(query, engine, params=params)
        
        print(f"✓ Query executada com sucesso! {len(df)} linhas retornadas.")
        return df
    
    except Exception as e:
        print(f"✗ Erro ao executar query: {e}")
        return pd.DataFrame()  # Retornar DataFrame vazio em caso de erro

print("✓ Função executar_query() criada!")

## 4️⃣ Explorar a Estrutura do Banco

Vamos ver quais tabelas existem no banco de dados:

In [ ]:
# Query para listar todas as tabelas do schema public
query_tabelas = """
SELECT 
    tablename as nome_tabela,
    schemaname as schema
FROM pg_tables 
WHERE schemaname = 'public'
ORDER BY tablename;
"""

df_tabelas = executar_query(query_tabelas)
print("\n📊 Tabelas disponíveis no banco de dados:")
print(df_tabelas)

### 📊 Contar registros em cada tabela

Vamos criar uma visão geral do volume de dados:

In [ ]:
# Dicionário para armazenar contagens
contagens = {}

# Lista de tabelas principais
tabelas_principais = [
    'vagas',
    'posicoes',
    'candidaturas',
    'talentos',
    'position_timeline',
    'candidatura_timeline',
    'requisicoes',
    'vaga_tags',
    'clientes'
]

# Contar registros em cada tabela
for tabela in tabelas_principais:
    query_count = f"SELECT COUNT(*) as total FROM {tabela};"
    df_count = executar_query(query_count)
    contagens[tabela] = df_count['total'].iloc[0]

# Criar DataFrame com as contagens
df_contagens = pd.DataFrame(list(contagens.items()), columns=['Tabela', 'Total de Registros'])
df_contagens = df_contagens.sort_values('Total de Registros', ascending=False).reset_index(drop=True)

print("\n📈 Volume de dados por tabela:")
print(df_contagens.to_string(index=False))

# Calcular total geral
total_geral = df_contagens['Total de Registros'].sum()
print(f"\n🎯 Total de registros no banco: {total_geral:,}")

## 5️⃣ Explorar Tabelas Principais

### 5.1 Tabela VAGAS

In [ ]:
# Carregar primeiras 10 vagas
query_vagas = """
SELECT 
    id,
    name,
    area,
    status,
    seniority,
    sla_days_goal,
    created_at_inhire,
    updated_at_inhire
FROM vagas
ORDER BY updated_at_inhire DESC
LIMIT 10;
"""

df_vagas_sample = executar_query(query_vagas)
print("\n📋 Amostra da tabela VAGAS (10 registros mais recentes):")
display(df_vagas_sample)

In [ ]:
# Ver estatísticas descritivas das vagas
print("📊 Informações sobre a tabela VAGAS:")
print(df_vagas_sample.info())

print("\n📈 Estatísticas descritivas:")
print(df_vagas_sample.describe(include='all'))

### 5.2 Distribuição de Status das Vagas

In [ ]:
# Query para contar vagas por status
query_status = """
SELECT 
    status,
    COUNT(*) as total,
    ROUND(COUNT(*)::numeric / (SELECT COUNT(*) FROM vagas) * 100, 2) as percentual
FROM vagas
GROUP BY status
ORDER BY total DESC;
"""

df_status = executar_query(query_status)
print("\n📊 Distribuição de status das vagas:")
print(df_status.to_string(index=False))

### 5.3 Tabela CANDIDATURAS

In [ ]:
# Carregar amostra de candidaturas
query_candidaturas = """
SELECT 
    c.id,
    c.talent_name,
    c.talent_email,
    c.source,
    c.status,
    c.stage_name,
    c.stage_order,
    v.name as vaga_nome,
    v.area as vaga_area
FROM candidaturas c
INNER JOIN vagas v ON c.vaga_id = v.id
ORDER BY c.updated_at DESC
LIMIT 10;
"""

df_candidaturas_sample = executar_query(query_candidaturas)
print("\n📋 Amostra da tabela CANDIDATURAS (10 registros mais recentes):")
display(df_candidaturas_sample)

## 6️⃣ Funções Helper Adicionais

Vamos criar mais algumas funções úteis para facilitar análises futuras:

In [ ]:
def get_tabela_completa(nome_tabela: str, limit: int = None) -> pd.DataFrame:
    """
    Retorna todos os registros de uma tabela (ou um limite)
    
    Parâmetros:
        nome_tabela (str): Nome da tabela a consultar
        limit (int): Limite de registros (None = todos)
    
    Retorna:
        pd.DataFrame: Dados da tabela
    """
    query = f"SELECT * FROM {nome_tabela}"
    if limit:
        query += f" LIMIT {limit}"
    
    return executar_query(query)


def get_colunas_tabela(nome_tabela: str) -> pd.DataFrame:
    """
    Retorna informações sobre as colunas de uma tabela
    
    Parâmetros:
        nome_tabela (str): Nome da tabela
    
    Retorna:
        pd.DataFrame: Informações das colunas
    """
    query = f"""
    SELECT 
        column_name as coluna,
        data_type as tipo,
        is_nullable as permite_null
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = '{nome_tabela}'
    ORDER BY ordinal_position;
    """
    
    return executar_query(query)


def get_estatisticas_basicas(tabela: str, coluna: str) -> dict:
    """
    Retorna estatísticas básicas de uma coluna numérica
    
    Parâmetros:
        tabela (str): Nome da tabela
        coluna (str): Nome da coluna
    
    Retorna:
        dict: Dicionário com estatísticas
    """
    query = f"""
    SELECT 
        COUNT(*) as total,
        COUNT({coluna}) as nao_nulos,
        MIN({coluna}) as minimo,
        MAX({coluna}) as maximo,
        AVG({coluna}) as media,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {coluna}) as mediana
    FROM {tabela};
    """
    
    df = executar_query(query)
    return df.to_dict('records')[0] if not df.empty else {}


print("✓ Funções helper criadas com sucesso!")
print("  - get_tabela_completa()")
print("  - get_colunas_tabela()")
print("  - get_estatisticas_basicas()")

### Testar funções helper:

In [ ]:
# Exemplo 1: Ver colunas da tabela vagas
print("📋 Colunas da tabela VAGAS:")
df_colunas_vagas = get_colunas_tabela('vagas')
display(df_colunas_vagas)

In [ ]:
# Exemplo 2: Estatísticas do SLA
print("📊 Estatísticas de SLA (dias):")
stats_sla = get_estatisticas_basicas('vagas', 'sla_days_goal')
for chave, valor in stats_sla.items():
    print(f"  {chave}: {valor}")

## 7️⃣ Exercícios Práticos (Opcional)

Agora é sua vez! Tente executar as seguintes queries:

### Exercício 1: Contar candidaturas por source
```sql
SELECT source, COUNT(*) as total
FROM candidaturas
GROUP BY source
ORDER BY total DESC;
```

In [ ]:
# Seu código aqui:
query_ex1 = """

"""

# df_ex1 = executar_query(query_ex1)
# display(df_ex1)

### Exercício 2: Encontrar vagas com SLA > 15 dias
```sql
SELECT id, name, sla_days_goal, status
FROM vagas
WHERE sla_days_goal > 15
ORDER BY sla_days_goal DESC
LIMIT 10;
```

In [ ]:
# Seu código aqui:
query_ex2 = """

"""

# df_ex2 = executar_query(query_ex2)
# display(df_ex2)

## ✅ Checkpoint: O que você aprendeu até aqui

- ✅ Como importar bibliotecas Python (pandas, numpy, psycopg2)
- ✅ Como conectar ao PostgreSQL usando SQLAlchemy
- ✅ Como executar queries SQL e carregar dados em DataFrames
- ✅ Como explorar a estrutura do banco de dados
- ✅ Como criar funções helper reutilizáveis
- ✅ Conceitos básicos de DataFrame, SQL e PostgreSQL

---

## 🎯 Próximos Passos

No próximo notebook (**02_eda_vagas_posicoes.ipynb**), vamos:
- Fazer análise exploratória das tabelas **vagas** e **posições**
- Criar visualizações (gráficos de barras, histogramas, time series)
- Calcular estatísticas descritivas
- Identificar padrões e insights iniciais

---

## 📚 Referências e Recursos Adicionais

**Pandas:**
- [Documentação Oficial](https://pandas.pydata.org/docs/)
- [10 Minutes to Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)

**SQL:**
- [W3Schools SQL Tutorial](https://www.w3schools.com/sql/)
- [PostgreSQL Documentation](https://www.postgresql.org/docs/)

**Python para Data Science:**
- [Real Python](https://realpython.com/)
- [Kaggle Learn](https://www.kaggle.com/learn)

---

**🎓 Parabéns! Você completou o Notebook 01!**